<a href="https://colab.research.google.com/github/deltorobarba/astrophysics/blob/main/ssds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sloan Digital Sky Survey (SDSS)**

[SDSS-V](https://www.sdss.org/) is the first facility providing multi-epoch optical & IR spectroscopy across the entire sky, as well as offering contiguous integral-field spectroscopic coverage of the Milky Way and Local Volume galaxies. This panoptic spectroscopic survey continues the strong SDSS legacy of innovative data and collaboration infrastructure.

In [ ]:
from google.cloud import batch_v1

def create_batch_job(project_id: str, region: str, job_name: str, bucket_name: str):
    """
    Creates a Batch job for processing astronomical data.

    Args:
        project_id: Google Cloud project ID.
        region: Google Cloud region for running the job.
        job_name: Unique name for the Batch job.
        bucket_name: Cloud Storage bucket containing the input data.
    """

    client = batch_v1.BatchServiceClient()
    job = batch_v1.Job()
    job.name = f"projects/{project_id}/locations/{region}/jobs/{job_name}"

    # Define a task group with one task.
    task_group = batch_v1.TaskGroup()
    task_group.task_count = 1

    # Define the task specification.
    task_spec = batch_v1.TaskSpec()

    # Use a public astronomical dataset from Cloud Storage.
    # Replace with your desired dataset and processing script.
    task_spec.runnables = [
        batch_v1.Runnable(
            container=batch_v1.Runnable.Container(
                image="gcr.io/google.com/cloudsdktool/cloud-sdk:latest",
                entrypoint=["/bin/sh", "-c"],
                commands=[
                    "gsutil cp gs://{bucket_name}/astronomical_data.fits .",
                    "python process_astronomical_data.py astronomical_data.fits",
                    "gsutil cp output_data.txt gs://{bucket_name}/output/",
                ],
            )
        )
    ]

    # Set the machine type and resources for the task.
    task_spec.compute_resource = batch_v1.ComputeResource(
        cpu_milli = 2000, # 2 vCPUs
        memory_mib = 8192 # 8 GB memory
    )

    task_group.task_spec = task_spec
    job.task_groups = [task_group]

    # Specify the Cloud Storage bucket for job logs.
    job.logs_policy = batch_v1.LogsPolicy(
        destination=batch_v1.LogsPolicy.Destination.CLOUD_LOGGING
    )

    create_request = batch_v1.CreateJobRequest(
        parent=f"projects/{project_id}/locations/{region}", job=job
    )
    created_job = client.create_job(request=create_request)
    print(f"Created job: {created_job.name}")

# Replace with your project ID, region, job name, and bucket name.
project_id = "your-project-id"
region = "us-central1"
job_name = "astronomical-data-job"
bucket_name = "your-bucket-name"

create_batch_job(project_id, region, job_name, bucket_name)

 * Import the Batch library: from google.cloud import batch_v1
 * create_batch_job function:
   * Takes your project ID, region, job name, and bucket name as input.
   * Initializes the Batch service client.
   * Creates a Job object with the specified name.
   * Defines a TaskGroup with one task.
   * Defines a TaskSpec with the following:
     * runnables: A list of commands to execute within a container.
       * Uses the gcr.io/google.com/cloudsdktool/cloud-sdk:latest image, which includes gsutil and python.
       * Copies the astronomical_data.fits file from your Cloud Storage bucket.
       * Runs a Python script (process_astronomical_data.py) to process the data.
       * Copies the output file (output_data.txt) back to your bucket.
     * compute_resource: Specifies the required CPU and memory for the task.
   * Sets the logs_policy to store job logs in Cloud Logging.
   * Creates the job using the Batch service client.
Before running the code:
 * Replace placeholders: Update the project_id, region, job_name, and bucket_name variables with your actual values.
 * Dataset:  Choose a public astronomical dataset from Cloud Storage or upload your own data. Update the bucket_name and file name accordingly.
 * Processing script: Create a Python script (process_astronomical_data.py) to process the data. This script should read the input file, perform the desired analysis, and write the output to output_data.txt.
 * Authentication: Ensure that your environment is authenticated to access Google Cloud resources. You can use the gcloud auth login command or set the GOOGLE_APPLICATION_CREDENTIALS environment variable.
Example dataset:
You can use the following public dataset from the Sloan Digital Sky Survey (SDSS) available on Google Cloud Storage:
 * Bucket: gcp-public-data-landsat
 * File: LC08_L1TP_047027_20200618_20200625_01_T1/LC08_L1TP_047027_20200618_20200625_01_T1_B4.TIF
This dataset contains a Landsat 8 image, which you can analyze using astronomical image processing techniques in your Python script.
This example demonstrates how to use Batch to process astronomical data. You can adapt this code for other scientific datasets and processing tasks by modifying the input data, processing script, and container image.
